# 7 — The marker DSL

The compute layer's `f`s are *declarations*, not callbacks: a name plus the
algebra a compiler can read. `mdsl.py` makes that set **open**. `defmarker`
traces a plain Python lambda into an owned expression-tree IR — three frozen
dataclasses, `Arg` / `Const` / `Prim` — and every consumer downstream walks
those trees: numpy evaluation, carrier/unit signatures, op counts, and
**derivatives**.

That last one is the point. Partial derivatives are DERIVED by tree
rewriting, so an activation function differentiates the day you write it.
The hand-maintained gradient table stops growing, and the
silent-zero-gradient class of bug dies structurally rather than by
vigilance.

The module imports nothing from the rest of tensorlib and nothing from the
main `pdum.dsl` package: the Node schema *is* the stability boundary. The
tracer below is one **producer** of Nodes; a lowered frontend AST would be
another. Consumers can never tell which — that is the no-rewrite guarantee,
held by a schema rather than by a promise.

In [1]:
import nbhelp  # noqa: F401  — puts tensorlib on sys.path
import numpy as np
from pdum.tl import Tensor, defmarker, defreducer, pointwise, reduce, scan, u
from pdum.tl.autodiff import grad, numeric_grad
from pdum.tl.ir import Instr, Program, run
from pdum.tl.mdsl import (
    COMPOSITE_MARKERS,
    COMPOSITE_REDUCERS,
    Arg,
    Const,
    diff,
    exp,
    gt,
    log,
    node_digest,
    tanh,
    trace,
    where,
)
from pdum.tl.opcount import ops_count

In [2]:
def instr(var, op, operands=(), **params):
    return Instr(var, op, tuple(operands), params)


def T(arr, names):
    return Tensor.from_numpy(np.asarray(arr, dtype=np.float64), names)


_INFIX = {"add": "+", "sub": "-", "mul": "*", "div": "/"}


def expr(n, names=("x", "y", "z", "w")):
    """Render a marker tree as infix — the trees are the whole story here."""
    if isinstance(n, Arg):
        return names[n.index] if n.index < len(names) else f"a{n.index}"
    if isinstance(n, Const):
        return f"{n.value}"
    if n.op in _INFIX:
        return f"({expr(n.args[0], names)} {_INFIX[n.op]} {expr(n.args[1], names)})"
    if n.op == "neg":
        return f"-{expr(n.args[0], names)}"
    return f"{n.op}({', '.join(expr(a, names) for a in n.args)})"


rng = np.random.default_rng(0)

## A lambda in, a tree out

`defmarker(name, arity, fn)` runs `fn` on symbolic scalars whose operators
build nodes instead of numbers. Nothing clever happens — the "compiler" is
operator overloading, about forty lines — and what comes back is data.

In [3]:
sigmoid = defmarker("sigmoid", 1, lambda x: 1 / (1 + exp(-x)))

print("the Node schema, verbatim:")
print(" ", sigmoid.body)
print()
print(f"{sigmoid.name}(x) = {expr(sigmoid.body)}   (arity {sigmoid.arity})")
print()
print("a smaller one:", trace(lambda x: 2 * x + 1, 1))

the Node schema, verbatim:
  Prim(op='div', args=(Const(value=1), Prim(op='add', args=(Prim(op='exp', args=(Prim(op='neg', args=(Arg(index=0),)),)), Const(value=1)))))

sigmoid(x) = (1 / (exp(-x) + 1))   (arity 1)

a smaller one: Prim(op='add', args=(Prim(op='mul', args=(Arg(index=0), Const(value=2))), Const(value=1)))


## `where` is the branch

Marker bodies have no control flow — matching the program IR's no-branching
rule. Tracing a Python `if` on a symbolic scalar raises instead of silently
baking in one side, and the fix is the same one the IR uses: make the branch
*data flow*.

In [4]:
try:
    defmarker("relu_bad", 1, lambda x: x if x else -x)
except TypeError as e:
    print("refused:", e)

relu = defmarker("relu", 1, lambda x: where(gt(x, 0), x, 0.0))
print()
print("relu(x) =", expr(relu.body))

refused: Python control flow cannot be traced into a marker body; use where(cond, a, b) — the branch is data flow here

relu(x) = where(gt(x, 0), x, 0.0)


## Composites evaluate like any other marker

`pointwise` accepts a `CompositeMarker` wherever it accepts a primitive: it
walks the tree over the aligned numpy arrays. Constants live in VALUE space,
so floats are honest here (0.5, √(2/π)) — the coordinates-exact /
values-inexact doctrine, applied.

In [5]:
softplus = defmarker("softplus", 1, lambda x: log(1 + exp(x)))
GELU_C = 0.7978845608028654  # sqrt(2/pi)
gelu = defmarker("gelu", 1, lambda x: 0.5 * x * (1 + tanh(GELU_C * (x + 0.044715 * x * x * x))))

xs = np.linspace(-3, 3, 13)
t = T(xs, ("i",))
refs = {
    "sigmoid": 1 / (1 + np.exp(-xs)),
    "softplus": np.log1p(np.exp(xs)),
    "gelu": 0.5 * xs * (1 + np.tanh(GELU_C * (xs + 0.044715 * xs**3))),
    "relu": np.maximum(xs, 0),
}
for m in (sigmoid, softplus, gelu, relu):
    got = pointwise(m, t).to_numpy()
    print(f"  {m.name:<9} max|Δ| vs numpy = {np.abs(got - refs[m.name]).max():.2e}")

  sigmoid   max|Δ| vs numpy = 0.00e+00
  softplus  max|Δ| vs numpy = 1.11e-16
  gelu      max|Δ| vs numpy = 0.00e+00
  relu      max|Δ| vs numpy = 0.00e+00


## Partials are derived, not declared

`m.partial(i)` rewrites the body into d(body)/d(Arg i) — chain rule applied
structurally, with light zero/one folding so the trees stay small — and
registers the result as *another composite marker*. Derivatives are
therefore first-class markers: differentiable again, evaluable, countable.

The trees are not algebraically simplified (no factoring, no
`s·(1−s)` recognition). They are numerically right, which is the contract;
recognizing that a subexpression is the forward output is an optimizer's
job, not the derivative rule's.

In [6]:
ds = sigmoid.partial(0)
d2 = ds.partial(0)  # a derivative of a derivative — nothing special about it
s = 1 / (1 + np.exp(-xs))
e1 = np.abs(pointwise(ds, t).to_numpy() - s * (1 - s)).max()
e2 = np.abs(pointwise(d2, t).to_numpy() - s * (1 - s) * (1 - 2 * s)).max()

print(f"{ds.name}(x) = {expr(ds.body)}")
print(f"      vs s(1-s):       max|Δ| = {e1:.2e}")
print()
print(f"{d2.name}(x) = {expr(d2.body)}")
print(f"      vs s(1-s)(1-2s): max|Δ| = {e2:.2e}")
print()
print("relu.d0(x) =", expr(relu.partial(0).body), " — the branch differentiates as a branch")
print()
print("registry:", sorted(COMPOSITE_MARKERS))

sigmoid.d0(x) = (-(1 / ((exp(-x) + 1) * (exp(-x) + 1))) * (exp(-x) * -1))
      vs s(1-s):       max|Δ| = 1.46e-16

sigmoid.d0.d0(x) = (((exp(-x) * -1) * (-1 * (-(1 / (((exp(-x) + 1) * (exp(-x) + 1)) * ((exp(-x) + 1) * (exp(-x) + 1)))) * (((exp(-x) + 1) * (exp(-x) * -1)) + ((exp(-x) + 1) * (exp(-x) * -1)))))) + (-(1 / ((exp(-x) + 1) * (exp(-x) + 1))) * (-1 * (exp(-x) * -1))))
      vs s(1-s)(1-2s): max|Δ| = 1.18e-16

relu.d0(x) = where(gt(x, 0), 1, 0)  — the branch differentiates as a branch

registry: ['gelu', 'relu', 'relu.d0', 'sigmoid', 'sigmoid.d0', 'sigmoid.d0.d0', 'softplus']


## …so gradients come for free

Composites resolve by name inside the IR (`f="gelu"`), and `grad` reaches
them through exactly one rule: *look up the composite, take its partials,
multiply by the incoming cotangent*. No entry per activation. Below, two
activations invented in the cell itself — `swish` and `mish` — differentiate
correctly on first use, validated against finite differences.

In [7]:
def validate(instrs, inputs, wrt, target=None):
    prog = Program(tuple(instrs))
    target = target or prog.instrs[-1].var
    joint, grads = grad(prog, target, dict(inputs))
    env = run(joint, inputs)
    got = env[grads[wrt]].to_numpy(order=inputs[wrt].names)
    want = numeric_grad(prog, target, wrt, inputs)
    ok = np.allclose(got, want, rtol=1e-4, atol=1e-6)
    print(("OK " if ok else "FAIL") + f"  d{target}/d{wrt}   max|Δ| = {np.abs(got - want).max():.2e}")
    return joint, grads, env


swish = defmarker("swish", 1, lambda x: x / (1 + exp(-x)))
mish = defmarker("mish", 1, lambda x: x * tanh(log(1 + exp(x))))

for m in (sigmoid, softplus, gelu, relu, swish, mish):
    p = [
        instr("x", "input"),
        instr("w", "input"),
        instr("p", "pointwise", ["x", "w"], f="mul"),
        instr("a", "pointwise", ["p"], f=m.name),
        instr("y", "reduce", ["a"], f="sum", dims=("i",)),
    ]
    inputs = {"x": T(rng.uniform(0.2, 1.5, 5), ("i",)), "w": T(rng.uniform(0.2, 1.5, 5), ("i",))}
    print(f"  {m.name:<9}", end=" ")
    validate(p, inputs, "x")

  sigmoid   OK   dy/dx   max|Δ| = 1.91e-10
  softplus  OK   dy/dx   max|Δ| = 2.95e-10
  gelu      OK   dy/dx   max|Δ| = 2.34e-10
  relu      OK   dy/dx   max|Δ| = 2.23e-10
  swish     OK   dy/dx   max|Δ| = 1.80e-10
  mish      OK   dy/dx   max|Δ| = 2.02e-10


The generated program says it plainly: the backward pass calls a marker
named `mish.d0` that nobody wrote.

In [8]:
p = [
    instr("x", "input"),
    instr("a", "pointwise", ["x"], f="mish"),
    instr("y", "reduce", ["a"], f="sum", dims=("i",)),
]
joint, grads, env = validate(p, {"x": T(rng.standard_normal(4), ("i",))}, "x")
print()
print(joint)

OK   dy/dx   max|Δ| = 1.29e-10

x = input()
a = pointwise(x; f='mish')
y = reduce(a; f='sum', dims=('i',))
%seed0 = const(; value=1.0, dims=())
%g1 = repeat(%seed0; name='i', extent=(0, 4), chart=None, labels=None)
%st2 = with_charts(%g1; charts={'i': None})
%g3 = pointwise(x; f='mish.d0')
%g4 = pointwise(%st2, %g3; f='mul')
%st5 = with_charts(%g4; charts={'i': None})


## Gradient-free positions, and signatures

Two disciplines ride along on the same trees. **Carriers**: comparisons
produce bools, so their derivative builders are `None` and the rewrite
folds the whole path to a constant zero — a mask contributes no cotangent
by construction, not by a special case in the AD pass. **Units**:
`marker_signature` walks the tree propagating carrier/unit facts, so a
transcendental applied to micrometers refuses.

In [9]:
mask = defmarker("mask", 2, lambda a, b: gt(a, b))
print("mask(x, y)  =", expr(mask.body))
print("d(mask)/dx  =", diff(mask.body, 0), " — comparisons carry no cotangent")
print()

try:
    pointwise(sigmoid, T(xs, ("i",)).with_value_units(u.um))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

mask(x, y)  = gt(x, y)
d(mask)/dx  = Const(value=0)  — comparisons carry no cotangent

SignatureError: exp: argument must be dimensionless, got um


## Reducers with structured state

`defreducer` is the other half: a fold whose accumulator is a **tuple**.
Four pieces — `lift` (element → state), `combine` (state ⊕ state,
associative), `init` (the identity state), `project` (state → scalar). A
one-component state is an ordinary reduction; `scan` keeps every prefix.

In [10]:
csum = defreducer(
    "csum",
    state=1,
    element=1,
    lift=lambda x: (x,),
    combine=lambda left, right: (left[0] + right[0],),
    init=(0.0,),
)
xv = rng.standard_normal(6)
print("scan  :", np.round(scan(csum, T(xv, ("i",)), "i").to_numpy(), 6))
print("cumsum:", np.round(np.cumsum(xv), 6))
print("reduce:", round(float(reduce(csum, T(xv, ("i",)), ("i",)).item()), 6), "= the final state")

scan  : [0.32897  0.070397 1.65387  2.974231 3.607584 1.404074]
cumsum: [0.32897  0.070397 1.65387  2.974231 3.607584 1.404074]
reduce: 1.404074 = the final state


Two components buy the shape that linear recurrences (SSMs) need. The
recurrence h_t = a_t·h_{t−1} + b_t is not a fold over a scalar — but it *is*
an associative fold over the pair (A, B) ↦ (aA, aB + b):

    (A₁,B₁) ⊕ (A₂,B₂) = (A₁·A₂, A₂·B₁ + B₂)

`lift` carries each timestep's (a_t, b_t) into the pair, `project` reads B
back out. `scan` then consumes **several aligned element tensors at once**
— the reduction's arity is the reducer's, not one.

In [11]:
linrec = defreducer(
    "linrec",
    state=2,
    element=2,
    lift=lambda a, b: (a, b),
    combine=lambda left, right: (left[0] * right[0], right[0] * left[1] + right[1]),
    init=(1.0, 0.0),
    project=lambda A, B: B,
)
SN = ("A", "B", "A2", "B2")
print("lift    :", tuple(expr(n, ("a", "b")) for n in linrec.lift))
print("combine :", tuple(expr(n, SN) for n in linrec.combine))
print("init    :", linrec.init, "  (the identity state: the empty reduction)")
print("project :", expr(linrec.project, ("A", "B")))

lift    : ('a', 'b')
combine : ('(A * A2)', '((A2 * B) + B2)')
init    : (1.0, 0.0)   (the identity state: the empty reduction)
project : B


In [12]:
n = 8
a = rng.uniform(0.5, 1.1, n)
b = rng.standard_normal(n)

h_ref, acc = np.empty(n), 0.0
for i in range(n):  # the recurrence, spelled out
    acc = a[i] * acc + b[i]
    h_ref[i] = acc

print("scan   :", np.round(scan(linrec, (T(a, ("t",)), T(b, ("t",))), "t").to_numpy(), 6))
print("loop   :", np.round(h_ref, 6))
print("reduce :", round(float(reduce(linrec, (T(a, ("t",)), T(b, ("t",))), ("t",)).item()), 6), "= the last state")
e = T(np.zeros(0), ("t",))
print("empty  :", float(reduce(linrec, (e, e), ("t",)).item()), "= project(init) — no special case needed")

scan   : [-0.661528  0.236283  0.229628  2.248727  1.98736   0.867536  0.379039
 -0.675319]
loop   : [-0.661528  0.236283  0.229628  2.248727  1.98736   0.867536  0.379039
 -0.675319]
reduce : -0.675319 = the last state
empty  : 0.0 = project(init) — no special case needed


## Associativity is declared, not verified

`associative=True` is a claim the library takes on faith and the compiler
spends: it is the license to evaluate the scan in O(log n) (Blelloch) rather
than the sequential sweep the reference layer runs. Nothing here checks it —
so property-test it now, and expect it to become a typeclass instance
obligation in the Lean model.

In [13]:
from pdum.tl.compute import _eval_tree  # the reference tree evaluator


def combine(s1, s2):
    return [_eval_tree(node, list(s1) + list(s2)) for node in linrec.combine]


probe = np.random.default_rng(3)
worst = 0.0
for _ in range(200):
    s1, s2, s3 = (list(probe.standard_normal(2)) for _ in range(3))
    lhs, rhs = combine(combine(s1, s2), s3), combine(s1, combine(s2, s3))
    worst = max(worst, float(np.abs(np.array(lhs) - np.array(rhs)).max()))

print("declared:", linrec.associative)
print(f"worst |(s1⊕s2)⊕s3 − s1⊕(s2⊕s3)| over 200 random triples: {worst:.2e}")

declared: True
worst |(s1⊕s2)⊕s3 − s1⊕(s2⊕s3)| over 200 random triples: 8.88e-16


## Differentiating the scan: BPTT emitted as IR

Backpropagation through a structured-state scan is the flagship. With
s_t = C(s_{t−1}, lift(e_t)) and y_t = P(s_t), the state cotangent obeys

    ŝ_t = Pᵀ(s_t)·ȳ_t + C_leftᵀ(s_t, l_{t+1})·ŝ_{t+1}

which is itself a **linear recurrence in reversed time** — so it runs as
another composite scan, generated on the spot (`linrec.adj0/1`, a matrix
pair-combine over k² + k state). Every Jacobian entry in it is a derived
partial of the very trees you wrote above. Nothing about `linrec` was
special-cased anywhere.

In [14]:
ssm = [
    instr("a", "input"),
    instr("b", "input"),
    instr("h", "scan", ["a", "b"], f="linrec", dim="t"),
    instr("hh", "pointwise", ["h", "h"], f="mul"),
    instr("y", "reduce", ["hh"], f="sum", dims=("t",)),
]
inputs = {"a": T(rng.uniform(0.4, 1.2, 6), ("t",)), "b": T(rng.standard_normal(6), ("t",))}
joint, grads, env = validate(ssm, inputs, "a")
joint, grads, env = validate(ssm, inputs, "b")
print()
print(f"{len(ssm)} forward instructions became {len(joint.instrs)}; the scans in the joint program:")
for i in joint.instrs:
    if i.op == "scan":
        print("  ", i)

OK   dy/da   max|Δ| = 2.86e-10
OK   dy/db   max|Δ| = 3.15e-10

5 forward instructions became 63; the scans in the joint program:
   h = scan(a, b; f='linrec', dim='t')
   %g11 = scan(%lat9, %lat10; f='linrec.s0', dim='t')
   %g12 = scan(%lat9, %lat10; f='linrec.s1', dim='t')
   %g41 = scan(%g28, %g30, %g32, %g34, %g37, %g40; f='linrec.adj0', dim='t')
   %g43 = scan(%g28, %g30, %g32, %g34, %g37, %g40; f='linrec.adj1', dim='t')


Two of those scans re-materialize the forward state trajectory (`linrec.s0`,
`linrec.s1` — the same fold projecting a different component; the reference
layer re-scans rather than caching, deliberately), and two run the backward
recurrence. The result matches the recurrence you would derive by hand: for
L = Σ_t h_t, the cotangent is h̄_t = 1 + a_{t+1}·h̄_{t+1}, and dL/db = h̄.

In [15]:
av, bv = np.array([0.9, 1.1, 0.7, 1.3, 0.8]), np.arange(1.0, 6.0)
lin = Program(
    (
        instr("a", "input"),
        instr("b", "input"),
        instr("h", "scan", ["a", "b"], f="linrec", dim="t"),
        instr("y", "reduce", ["h"], f="sum", dims=("t",)),
    )
)
ins = {"a": T(av, ("t",)), "b": T(bv, ("t",))}
jp, gr = grad(lin, "y", ins)
out = run(jp, ins)

hbar = np.empty(5)
hbar[-1] = 1.0
for i in range(3, -1, -1):
    hbar[i] = 1.0 + av[i + 1] * hbar[i + 1]

print("dL/db  (generated IR) :", np.round(out[gr["b"]].to_numpy(), 6))
print("h̄      (by hand)      :", np.round(hbar, 6))

dL/db  (generated IR) : [4.6718 3.338  3.34   1.8    1.    ]
h̄      (by hand)      : [4.6718 3.338  3.34   1.8    1.    ]


Everything the backward pass needed, it derived — and registered by name.
Here is what `linrec` grew into (`C` = combine components, `L` = lift, `P` =
project, `.dN` = partials, `.sN` = the state scanners, `.adjN` = the
generated backward recurrences):

In [16]:
def argnames(name):
    return SN if ".C" in name else ("a", "b") if ".L" in name else ("A", "B")


for name in sorted(COMPOSITE_MARKERS):
    if name.startswith("linrec"):
        print(f"  {name:<14} = {expr(COMPOSITE_MARKERS[name].body, argnames(name))}")
print()
for name in sorted(COMPOSITE_REDUCERS):
    if name.startswith("linrec"):
        r = COMPOSITE_REDUCERS[name]
        s = tuple(f"s{i}" for i in range(r.state))
        print(f"  {name:<14} state={r.state} element={r.element} project={expr(r.project, s)}")

  linrec.C0      = (A * A2)
  linrec.C0.d0   = A2
  linrec.C0.d1   = 0
  linrec.C0.d2   = A
  linrec.C0.d3   = 0
  linrec.C1      = ((A2 * B) + B2)
  linrec.C1.d0   = 0
  linrec.C1.d1   = A2
  linrec.C1.d2   = B
  linrec.C1.d3   = 1
  linrec.L0      = a
  linrec.L0.d0   = 1
  linrec.L0.d1   = 0
  linrec.L1      = b
  linrec.L1.d0   = 0
  linrec.L1.d1   = 1
  linrec.P       = B
  linrec.P.d0    = 0
  linrec.P.d1    = 1

  linrec         state=2 element=2 project=s1
  linrec.adj0    state=6 element=6 project=s4
  linrec.adj1    state=6 element=6 project=s5
  linrec.s0      state=2 element=2 project=s0
  linrec.s1      state=2 element=2 project=s1


The generated backward reducer is worth one look: its state is a k×k matrix
followed by a k-vector, and its combine is (M,v) ⊕ (N,w) = (N·M, N·v + w) —
matrix algebra, associative for free. The identity state is exactly the
identity matrix and a zero vector. This is the generic carrier for *any*
structured-state scan's backward pass; `linrec` just happens to be k = 2.

In [17]:
adj = COMPOSITE_REDUCERS["linrec.adj0"]
left = ("M00", "M01", "M10", "M11", "v0", "v1")
right = ("N00", "N01", "N10", "N11", "w0", "w1")
print("state:", adj.state, " init:", adj.init, " (identity matrix ++ zero vector)")
for i, node in enumerate(adj.combine):
    print(f"  out{i} =", expr(node, left + right))

state: 6  init: (1.0, 0.0, 0.0, 1.0, 0.0, 0.0)  (identity matrix ++ zero vector)
  out0 = ((N00 * M00) + (N01 * M10))
  out1 = ((N00 * M01) + (N01 * M11))
  out2 = ((N10 * M00) + (N11 * M10))
  out3 = ((N10 * M01) + (N11 * M11))
  out4 = (((N00 * v0) + (N01 * v1)) + w0)
  out5 = (((N10 * v0) + (N11 * v1)) + w1)


## Training through the scan

Which means gradient descent through a recurrence is just… running the
generated program in a loop. Recover the decay `a` of a linear recurrence
from its output, starting from 0.1:

In [18]:
N, TRUE_A = 32, 0.8
bseq = rng.standard_normal(N)
target, acc = np.empty(N), 0.0
for i in range(N):
    acc = TRUE_A * acc + bseq[i]
    target[i] = acc

fit = Program(
    (
        instr("a0", "input"),
        instr("b", "input"),
        instr("yt", "input"),
        instr("ar", "repeat", ["a0"], name="t", extent=(0, N)),
        instr("h", "scan", ["ar", "b"], f="linrec", dim="t"),
        instr("r", "pointwise", ["h", "yt"], f="sub"),
        instr("r2", "pointwise", ["r", "r"], f="mul"),
        instr("L", "reduce", ["r2"], f="mean", dims=("t",)),
    )
)


def feed(a0):
    return {"a0": T(np.float64(a0), ()), "b": T(bseq, ("t",)), "yt": T(target, ("t",))}


a0 = 0.1
jp, gr = grad(fit, "L", feed(a0))
for step in range(41):
    out = run(jp, feed(a0))
    g = float(out[gr["a0"]].item())
    if step % 10 == 0:
        print(f"step {step:3d}   a = {a0:.4f}   loss = {float(out['L'].item()):.6f}   dL/da = {g:+.4f}")
    a0 -= 0.02 * g
print("learned a =", round(a0, 4), "   true a =", TRUE_A)

step   0   a = 0.1000   loss = 1.647462   dL/da = -1.6531
step  10   a = 0.5072   loss = 0.774605   dL/da = -2.8208


step  20   a = 0.8000   loss = 0.000000   dL/da = +0.0002


step  30   a = 0.8000   loss = 0.000000   dL/da = +0.0000


step  40   a = 0.8000   loss = 0.000000   dL/da = +0.0000
learned a = 0.8    true a = 0.8


## Costs are the tree's, not a black box's

Because a composite is data, `ops_count` prices it by walking it: `gelu` is
not one opaque "gelu op" but the muls, adds and `tanh` it actually is. The
counts stay keyed by primitive NAME — what a `tanh` costs is a machine's
opinion, and belongs in a cost model applied afterwards.

In [19]:
prog = Program(
    (
        instr("x", "input"),
        instr("g", "pointwise", ["x"], f="gelu"),
        instr("y", "reduce", ["g"], f="sum", dims=("i",)),
    )
)
oc = ops_count(prog, {"x": T(np.zeros(6), ("i",)).layout})
print("gelu over 6 elements:", dict(oc.total))
print("per instruction     :", {k: dict(v) for k, v in oc.per_var.items() if v})

gelu over 6 elements: {'mul': 36, 'add': 17, 'tanh': 6}
per instruction     : {'g': {'mul': 36, 'add': 12, 'tanh': 6}, 'y': {'add': 5}}


## Naming is optional: the registry as a cache

Pass `name=None` and the name is content-addressed from the traced tree.
Re-tracing the same body — in a loop, in another module, from another
frontend — lands on the same registry entry rather than a second one. The
registry behaves as a cache, not a namespace.

In [20]:
anon = defmarker(None, 1, lambda x: 1 / (1 + exp(-x)))
print("anonymous name      :", anon.name)
print("digest == sigmoid's :", node_digest(anon.body) == node_digest(sigmoid.body))
print("re-tracing dedupes  :", anon is defmarker(None, 1, lambda z: 1 / (1 + exp(-z))))

anonymous name      : m_e4c83ff178
digest == sigmoid's : True
re-tracing dedupes  : True


---
Notes and known gaps (CONCERNS #17, #21). Associativity is declared and
property-tested, never verified — the Lean model turns it into an instance
obligation, and the O(log n) parallel evaluation it licenses is the
compiler's job (the reference sweep here is a sequential Python loop). The
backward pass re-scans the forward trajectory once per state component
instead of caching it: reference inefficiency, deliberate. Composite
reducers fold one dim at a time; reverse scan stays flip∘scan∘flip. The
registries are global and keyed by name, so re-registering a composite with
a changed body under an old name is the one sharp edge — content-addressed
names (`name=None`) avoid it entirely. And the derived trees are honest, not
clever: no common-subexpression sharing, no algebraic simplification. Both
are optimizer passes over data that is now, finally, available to optimize.